# FedUCA Extra-A: FedProx Baseline + Scalability (K=4,10,20) + 5-fold CV

**Add data:** `ham10000-vit-embeddings-orig` | **GPU T4** | Internet OFF

Adds three missing pieces for Q1:
1. FedProx as heterogeneity-aware baseline
2. Scalability: K=10 and K=20 clients
3. 5-fold lesion-grouped cross-validation

In [ ]:
import os, json, time, copy, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_extra_a"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7
EMBED_DIM   = 768

hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("Add ham10000-vit-embeddings-orig dataset first.")
EMBED_DIR = os.path.dirname(hits[0])
print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")

In [ ]:
# Load embeddings + lesion IDs
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    embs = torch.tensor(d['embeddings'], dtype=torch.float32)
    lbls = torch.tensor(d['labels'],     dtype=torch.long)
    ids  = d['lesion_ids']
    return embs, lbls, ids

emb_train, lbl_train, ids_train = load_split("train")
emb_val,   lbl_val,   ids_val   = load_split("val")
emb_test,  lbl_test,  ids_test  = load_split("test")

# Combine train+val for cross-validation
emb_tv  = torch.cat([emb_train, emb_val],  dim=0)
lbl_tv  = torch.cat([lbl_train, lbl_val],  dim=0)
ids_tv  = np.concatenate([ids_train, ids_val])

print(f"Train: {emb_train.shape} | Val: {emb_val.shape} | Test: {emb_test.shape}")
print(f"Train+Val combined: {emb_tv.shape}")

counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = (1.0/counts); weights = (weights/weights.mean()).to(DEVICE)
print(f"Class weights: {weights.cpu().numpy().round(3)}")

In [ ]:
# Shared model + utilities
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768,4*b), nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b,b),   nn.GELU(), nn.LayerNorm(b),   nn.Dropout(0.2),
            nn.Linear(b,4*b),   nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b,7))
    def forward(self, x): return self.net(x)

def dirichlet_partition(labels, K, alpha, seed):
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max())+1):
        ci = np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval(); p, t = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p = torch.cat(p).numpy(); t = torch.cat(t).numpy()
    f1s = []
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
        f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy":float((p==t).mean()), "macro_f1":float(np.mean(f1s)),
            "f1_per_class":[float(x) for x in f1s]}

def fedavg(states, sizes):
    total = sum(sizes)
    return {k: sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

print("Utilities ready.")

In [ ]:
# ── LOCAL TRAINING: Standard FedAvg ──────────────────────────────────
def local_train_fedavg(head, embs, labels, lr=5e-3, epochs=5, bs=32):
    h = copy.deepcopy(head).to(DEVICE); h.train()
    opt  = optim.Adam(h.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        for x, y in DataLoader(TensorDataset(embs.to(DEVICE), labels.to(DEVICE)),
                               batch_size=bs, shuffle=True):
            opt.zero_grad(); crit(h(x), y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

# ── LOCAL TRAINING: FedProx ───────────────────────────────────────────
def local_train_fedprox(head, embs, labels, global_state,
                         mu=0.01, lr=5e-3, epochs=5, bs=32):
    """
    FedProx: adds proximal term mu/2 * ||w - w_global||^2 to local loss.
    Penalises local weights from drifting far from the global model.
    mu=0.01 is the standard value from Li et al. 2020.
    """
    h = copy.deepcopy(head).to(DEVICE); h.train()
    opt  = optim.Adam(h.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=weights)
    # Move global weights to device for proximal term
    global_params = {k: v.to(DEVICE) for k, v in global_state.items()}
    for _ in range(epochs):
        for x, y in DataLoader(TensorDataset(embs.to(DEVICE), labels.to(DEVICE)),
                               batch_size=bs, shuffle=True):
            opt.zero_grad()
            ce_loss = crit(h(x), y)
            # Proximal term
            prox = torch.tensor(0., device=DEVICE)
            for name, param in h.named_parameters():
                if name in global_params:
                    prox += (param - global_params[name]).pow(2).sum()
            loss = ce_loss + (mu / 2) * prox
            loss.backward(); opt.step()
    return copy.deepcopy(h.state_dict())

print("FedAvg and FedProx local training functions ready.")
print("FedProx mu=0.01 (Li et al. 2020 default)")

In [ ]:
# ── PGA AGGREGATION (same as Phase 3) ────────────────────────────────
def compute_prototypes(embs, labels):
    proto = torch.zeros(NUM_CLASSES, EMBED_DIM)
    count = torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask = labels == c
        if mask.sum() > 0:
            proto[c] = embs[mask].mean(0)
            count[c] = float(mask.sum())
    return proto, count

def prototype_guided_aggregation(states, sizes, client_protos, beta=5.0):
    K = len(states); total = sum(sizes)
    g_proto = torch.zeros(NUM_CLASSES, EMBED_DIM)
    g_count = torch.zeros(NUM_CLASSES)
    for k in range(K):
        pk, ck = client_protos[k]
        for c in range(NUM_CLASSES):
            if ck[c] > 0:
                g_proto[c] += ck[c]*pk[c]; g_count[c] += ck[c]
    for c in range(NUM_CLASSES):
        if g_count[c] > 0: g_proto[c] /= g_count[c]
    cos_d = lambda a,b: 1.0 - float(
        (a/(a.norm()+1e-9))@(b/(b.norm()+1e-9)))
    w = torch.zeros(NUM_CLASSES, K)
    for c in range(NUM_CLASSES):
        if g_proto[c].norm() < 1e-9:
            for k in range(K): w[c,k]=sizes[k]/total
        else:
            lw = torch.zeros(K)
            for k in range(K):
                pk,ck = client_protos[k]
                lw[k] = -beta*cos_d(pk[c],g_proto[c]) if ck[c]>0 else -1e6
            w[c] = torch.softmax(lw, dim=0)
    fw_key = fb_key = None
    for key in states[0]:
        s = states[0][key].shape
        if len(s)==2 and s[0]==NUM_CLASSES: fw_key=key
        if len(s)==1 and s[0]==NUM_CLASSES: fb_key=key
    avg = {}
    for key in states[0]:
        if key == fw_key:
            avg[key] = torch.stack([
                sum(w[c,k]*states[k][key][c].float() for k in range(K))
                for c in range(NUM_CLASSES)])
        elif key == fb_key:
            avg[key] = torch.stack([
                sum(w[c,k]*states[k][key][c].float() for k in range(K))
                for c in range(NUM_CLASSES)])
        else:
            avg[key] = sum((sizes[i]/total)*states[i][key].float()
                          for i in range(K))
    return avg

print("PGA aggregation ready.")

In [ ]:
# ── UNIFIED TRAINING LOOP ─────────────────────────────────────────────
def run_experiment(method, K, alpha, seed,
                   n_rounds=60, epochs=5, lr=5e-3, bs=32,
                   fedprox_mu=0.01, pga_beta=5.0, verbose=True):
    """
    method: 'fedavg' | 'fedprox' | 'pga'
    K     : number of clients (4, 10, or 20)
    """
    partition = dirichlet_partition(lbl_train, K, alpha, seed)
    c_embs    = [emb_train[idx] for idx in partition]
    c_labels  = [lbl_train[idx] for idx in partition]
    sizes     = [len(idx)       for idx in partition]

    # Pre-compute prototypes (PGA only)
    if method == 'pga':
        c_protos = [compute_prototypes(c_embs[k], c_labels[k])
                    for k in range(K)]

    head    = BottleneckMLP(ratio=4).to(DEVICE)
    history = []; best_val=-1; best_round=0; best_state=None
    t0 = time.time()

    for rnd in range(1, n_rounds+1):
        global_state = copy.deepcopy(head.state_dict())

        # Local training
        states = []
        for k in range(K):
            if method == 'fedprox':
                st = local_train_fedprox(head, c_embs[k], c_labels[k],
                                         global_state, fedprox_mu, lr, epochs, bs)
            else:
                st = local_train_fedavg(head, c_embs[k], c_labels[k],
                                        lr, epochs, bs)
            states.append(st)

        # Aggregation
        if method == 'pga':
            head.load_state_dict(
                prototype_guided_aggregation(states, sizes, c_protos, pga_beta))
        else:
            head.load_state_dict(fedavg(states, sizes))

        vm = evaluate(head, emb_val.to(DEVICE), lbl_val)
        history.append(vm['macro_f1'])
        if vm['macro_f1'] > best_val:
            best_val=vm['macro_f1']; best_round=rnd
            best_state=copy.deepcopy(head.state_dict())
        if verbose and (rnd%10==0 or rnd==1):
            print(f"  [{method} K={K}] r{rnd:3d} val={vm['macro_f1']:.4f} "
                  f"best={best_val:.4f} | {time.time()-t0:.0f}s")

    head.load_state_dict(best_state)
    tm = evaluate(head, emb_test.to(DEVICE), lbl_test)
    return {"method":method,"K":K,"alpha":alpha,"seed":seed,
            "best_round":best_round,"best_val_f1":best_val,
            "val_history":history,"test":tm}

print("run_experiment() ready — handles FedAvg, FedProx, PGA.")

In [ ]:
# ── EXPERIMENT A: FedProx vs FedAvg vs PGA at K=4 ────────────────────
print("="*60)
print("EXPERIMENT A: FedProx baseline (K=4, all alpha, 5 seeds)")
print("="*60)

ALPHAS=[1000,1,0.1]; SEEDS=[0,1,2,3,4]
results_a = []

for alpha in ALPHAS:
    print(f"\n--- alpha={alpha} ---")
    for method in ['fedprox']:   # fedavg+pga already done in P2/P3
        for seed in SEEDS:
            print(f"  {method} K=4 alpha={alpha} seed={seed}")
            res = run_experiment(method, K=4, alpha=alpha, seed=seed)
            results_a.append(res)
            with open(os.path.join(RESULTS_DIR,
                      f"{method}_k4_a{alpha}_s{seed}.json"),"w") as f:
                json.dump(res,f,indent=2)
        f1s=[r['test']['macro_f1'] for r in results_a
             if r['method']==method and r['alpha']==alpha]
        print(f"  >>> {method} alpha={alpha}: F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\nExperiment A complete.")

In [ ]:
# ── EXPERIMENT B: Scalability K=10 and K=20 ──────────────────────────
print("="*60)
print("EXPERIMENT B: Scalability — K=10 and K=20 clients")
print("="*60)

results_b = []

for K in [10, 20]:
    print(f"\n=== K={K} clients ===")
    for method in ['fedavg', 'pga']:
        for alpha in ALPHAS:
            for seed in SEEDS:
                print(f"  {method} K={K} alpha={alpha} seed={seed}")
                res = run_experiment(method, K=K, alpha=alpha, seed=seed,
                                     verbose=False)
                results_b.append(res)
                with open(os.path.join(RESULTS_DIR,
                          f"{method}_k{K}_a{alpha}_s{seed}.json"),"w") as f:
                    json.dump(res,f,indent=2)
            f1s=[r['test']['macro_f1'] for r in results_b
                 if r['method']==method and r['K']==K and r['alpha']==alpha]
            print(f"  {method} K={K} alpha={alpha}: F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\nExperiment B complete.")

In [ ]:
# ── EXPERIMENT C: 5-fold lesion-grouped cross-validation ─────────────
print("="*60)
print("EXPERIMENT C: 5-fold Lesion-Grouped Cross-Validation")
print("="*60)

results_cv = []
K_cv = 4; ALPHA_CV = 1000

# GroupKFold on lesion IDs (prevents leakage across folds)
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(
    np.arange(len(emb_tv)), lbl_tv.numpy(), groups=ids_tv))

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f"\n--- Fold {fold+1}/5 ---")
    fold_emb_tr = emb_tv[tr_idx]; fold_lbl_tr = lbl_tv[tr_idx]
    fold_emb_va = emb_tv[va_idx]; fold_lbl_va = lbl_tv[va_idx]

    # Recompute class weights per fold
    fold_counts  = torch.bincount(fold_lbl_tr, minlength=NUM_CLASSES).float()
    fold_weights = (1.0/fold_counts)
    fold_weights = (fold_weights/fold_weights.mean()).to(DEVICE)

    for method in ['fedavg', 'pga']:
        partition = dirichlet_partition(fold_lbl_tr, K_cv, ALPHA_CV, seed=fold)
        c_embs    = [fold_emb_tr[idx] for idx in partition]
        c_labels  = [fold_lbl_tr[idx] for idx in partition]
        sizes     = [len(idx)          for idx in partition]
        if method=='pga':
            c_protos = [compute_prototypes(c_embs[k],c_labels[k])
                        for k in range(K_cv)]

        # Temporarily override global weights for this fold
        old_weights = weights.clone()
        weights.data = fold_weights.data

        head = BottleneckMLP(ratio=4).to(DEVICE)
        best_val=-1; best_state=None
        for rnd in range(1, 61):
            global_state = copy.deepcopy(head.state_dict())
            states=[]
            for k in range(K_cv):
                st=local_train_fedavg(head,c_embs[k],c_labels[k])
                states.append(st)
            if method=='pga':
                head.load_state_dict(
                    prototype_guided_aggregation(states,sizes,c_protos))
            else:
                head.load_state_dict(fedavg(states,sizes))
            vm=evaluate(head,fold_emb_va.to(DEVICE),fold_lbl_va)
            if vm['macro_f1']>best_val:
                best_val=vm['macro_f1']
                best_state=copy.deepcopy(head.state_dict())

        weights.data = old_weights.data  # restore
        head.load_state_dict(best_state)
        vm_test = evaluate(head, emb_test.to(DEVICE), lbl_test)
        res = {"method":method,"fold":fold,"K":K_cv,"alpha":ALPHA_CV,
               "val_f1":best_val,"test":vm_test}
        results_cv.append(res)
        print(f"  {method} fold={fold+1}: val={best_val:.3f} "
              f"test={vm_test['macro_f1']:.3f}")

# Summary
print("\nCROSS-VALIDATION SUMMARY:")
for method in ['fedavg','pga']:
    vals = [r['val_f1']           for r in results_cv if r['method']==method]
    tests= [r['test']['macro_f1'] for r in results_cv if r['method']==method]
    print(f"  {method}: val={np.mean(vals):.3f}±{np.std(vals):.3f} "
          f"test={np.mean(tests):.3f}±{np.std(tests):.3f}")

# Save
with open(os.path.join(RESULTS_DIR,"cv_results.json"),"w") as f:
    json.dump(results_cv,f,indent=2)
with open(os.path.join(RESULTS_DIR,"scalability_results.json"),"w") as f:
    json.dump(results_b,f,indent=2)
with open(os.path.join(RESULTS_DIR,"fedprox_results.json"),"w") as f:
    json.dump(results_a,f,indent=2)
print("\nAll results saved.")

In [ ]:
# ── FINAL COMPARISON TABLE ────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL COMPARISON: FedAvg vs FedProx vs PGA")
print("(K=4, alpha=0.1 — the critical non-IID setting)")
print("="*65)

import json as _json
# Load P2/P3 results if available
try:
    p2_hits = glob.glob("/kaggle/input/**/all_results.json", recursive=True)
    with open(p2_hits[0]) as f: p2=_json.load(f)
    p3_hits = glob.glob("/kaggle/input/**/pga_all.json", recursive=True)
    with open(p3_hits[0]) as f: p3=_json.load(f)
    fa_f1s=[r['test']['macro_f1'] for r in p2
            if r['alpha']==0.1 and r['head_type']=='bottleneck' and r['ratio']==4]
    pg_f1s=[r['test']['macro_f1'] for r in p3 if r['alpha']==0.1]
    print(f"  FedAvg  K=4  alpha=0.1: F1={np.mean(fa_f1s):.3f}±{np.std(fa_f1s):.3f}")
    print(f"  PGA     K=4  alpha=0.1: F1={np.mean(pg_f1s):.3f}±{np.std(pg_f1s):.3f}")
except: pass

fp_f1s=[r['test']['macro_f1'] for r in results_a
        if r['alpha']==0.1 and r['method']=='fedprox']
if fp_f1s:
    print(f"  FedProx K=4  alpha=0.1: F1={np.mean(fp_f1s):.3f}±{np.std(fp_f1s):.3f}")

print("\nScalability summary (alpha=0.1):")
for K in [10,20]:
    for method in ['fedavg','pga']:
        f1s=[r['test']['macro_f1'] for r in results_b
             if r['K']==K and r['alpha']==0.1 and r['method']==method]
        if f1s:
            print(f"  {method:<8} K={K:<3} alpha=0.1: "
                  f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\nSave output as dataset: feduca-extra-a-results")
print("Then run Extra-B notebook for ISIC2019 generalizability.")